# RecruitRadar-IL — Detecting Iranian Recruitment via Telegram

Research notebook for collecting **public** Telegram channel data and surfacing
posts that match documented patterns of Iranian recruitment of Israelis
(low-effort "missions" for fast cash, crypto payment, photographing sites,
graffiti / poster-hanging, moving contact to private apps).

The notebook is an end-to-end, **runnable** pipeline:

1. Setup & configuration (credentials from `.env` or interactive prompts)
2. Connect to Telegram (Telethon)
3. Channel registry — jobs / freelance / marketplaces / rides / crypto
   exchangers in **Hebrew, Russian and English** (+ `channels_extra.txt`)
4. SQLite storage + raw-JSON archive
5. Privacy: SHA-256(+salt) hashing of every user id
6. Rate-limited bulk collection with `FloodWaitError` handling
7. Plain-text snapshot database (`data/txt/`) — diffable over time
8. EDA
9. Weak supervision — trilingual rule lexicon → noisy labels
10. Behavioral features + suspicion score
11. Appearance anomalies — per-channel Isolation Forest (what looks *odd*
    in this group even if no keyword matches)
12. Dashboard (`exports/dashboard.html`) + anonymized export

> Run cells top-to-bottom. Sections 2 and 6 talk to Telegram; everything from
> section 7 onward works offline on whatever you've already collected.

## ⚖️ Read first — legal & ethical ground rules

This tool collects only **public** channels/groups, the same data any member
sees. Operate strictly as a researcher documenting, **not** as an investigator.

- **Public only.** No private chats, no groups you weren't invited to.
- **No contact.** Never message, bait, or engage a suspected account.
- **No raw PII in the repo.** User ids are hashed (SHA-256 + salt) before
  anything leaves this machine. Phone numbers / usernames are never exported.
- **Data stays local.** `data/`, `*.session`, and `.env` are gitignored.
- **Document provenance.** Keep raw JSON so you can always show what you saw.
- **High precision over recall.** A false accusation has real consequences;
  treat every hit as a lead to review, never a conclusion.
- **Escalate responsibly.** Share findings with the relevant authorities /
  academic cyber units before any public release.

## 1. Setup & configuration

In [ ]:
# Install dependencies (safe to re-run; comment out once installed).
%pip install -q telethon pandas numpy matplotlib python-dotenv scikit-learn


In [ ]:
import os
import json
import hashlib
import sqlite3
import asyncio
import secrets
import getpass
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
import numpy as np

# Load any credentials already present in the environment / .env (all optional).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("python-dotenv not installed; reading from the real environment only.")


def _ask(prompt_text, secret=False):
    """Prompt on stdin (getpass hides secrets). Returns '' on empty input, EOF,
    or a kernel with no stdin (headless run), so a missing answer can never
    kill the kernel."""
    try:
        return (getpass.getpass(prompt_text) if secret else input(prompt_text)).strip()
    except Exception:
        print(f"  (no input available for: {prompt_text.strip()})")
        return ""


# --- Credentials -------------------------------------------------------------
# Priority for every value: environment / .env  ->  interactive prompt.
# Nothing is hardcoded, and a missing value NEVER raises — the live-collection
# cells simply stay disabled and the offline pipeline still runs.
API_ID   = os.getenv("TELEGRAM_API_ID")   or _ask("Telegram API ID  (my.telegram.org/apps): ")
API_HASH = os.getenv("TELEGRAM_API_HASH") or _ask("Telegram API HASH (kept hidden): ", secret=True)
PHONE    = os.getenv("TELEGRAM_PHONE")    or _ask("Telegram phone   (intl format, e.g. +9725...): ")

# Pseudonymization salt — a stable salt gives stable pseudonyms across runs.
HASH_SALT = os.getenv("HASH_SALT", "")
if not HASH_SALT or HASH_SALT == "change_me_to_a_long_random_string":
    HASH_SALT = _ask("HASH_SALT (blank = generate a random one for THIS session): ", secret=True)
    if not HASH_SALT:
        HASH_SALT = secrets.token_hex(32)
        print("⚠️  Generated a random HASH_SALT for this session only — pseudonyms will\n"
              "    NOT match earlier/later runs. Put a fixed HASH_SALT in .env to keep\n"
              "    them stable (python -c \"import secrets; print(secrets.token_hex(32))\").")

# Validate API_ID without crashing.
API_ID_INT = None
if API_ID:
    try:
        API_ID_INT = int(str(API_ID).strip())
    except ValueError:
        print("⚠️  API_ID is not a number — Telegram login will stay disabled until fixed.")

CREDENTIALS_OK = bool(API_ID_INT and API_HASH and PHONE)

# Project paths
DATA_DIR    = Path("data")
RAW_DIR     = DATA_DIR / "raw"          # raw JSON archive, one file per channel
TXT_DIR     = DATA_DIR / "txt"          # plain-text snapshot DB, one file per channel
EXPORT_DIR  = Path("exports")           # anonymized outputs + dashboard
DB_PATH     = DATA_DIR / "recruitradar.db"
for d in (DATA_DIR, RAW_DIR, TXT_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

if CREDENTIALS_OK:
    print(f"Config loaded. API_ID: {API_ID_INT} | phone: {PHONE} | DB: {DB_PATH}")
else:
    print("ℹ️  Telegram credentials incomplete — sections 2 & 6 (login + live\n"
          "   collection) will be SKIPPED. You can still run sections 7+ on demo\n"
          "   or previously-collected data. Re-run this cell any time to add them.")


In [ ]:
def hash_user_id(user_id) -> str:
    """SHA-256(salt + user_id). Stable per-salt so the same user maps to the
    same pseudonym across channels, but the real id is never stored/exported."""
    if user_id is None:
        return ""
    return hashlib.sha256((HASH_SALT + str(user_id)).encode("utf-8")).hexdigest()[:16]

# sanity check
print("example pseudonym:", hash_user_id(123456789))


## 2. Connect to Telegram

Section 1 collects your credentials from `.env`/environment **or** prompts you
for them interactively (API id, API hash, phone). This cell then logs in:
Telethon prompts for the **login code Telegram sends to your app** (not SMS),
and for your **two-step (2FA) password** if enabled. On success it writes a
`recruitradar.session` file (gitignored) so later runs are silent.

Everything is fault-tolerant: if a credential is missing, the code is wrong, or
you just press Enter, the cell prints a message and leaves `client = None`
instead of crashing — the offline pipeline (sections 7–10) still runs. Re-run
this cell any time to retry the login.

In Jupyter we use the **async** client directly with `await` (the notebook
already runs an event loop, so don't call `asyncio.run`).


In [ ]:
client = None                   # always defined, even if Telethon is missing
SESSION_NAME = "recruitradar"   # -> recruitradar.session (gitignored)

try:
    from telethon import TelegramClient
    from telethon.errors import (
        FloodWaitError, ChannelPrivateError, UsernameNotOccupiedError,
        SessionPasswordNeededError, PhoneCodeInvalidError, PhoneCodeExpiredError,
        PhoneNumberInvalidError,
    )
    TELETHON_OK = True
except ImportError:
    TELETHON_OK = False
    print("Telethon not installed — live collection disabled. "
          "`pip install telethon` to enable it; sections 7+ still run offline.")


async def telegram_login():
    """Interactive, fault-tolerant login.

    Prompts for the login code Telegram sends to your app and, if 2FA is on,
    your two-step password. Every failure path returns None instead of raising,
    so a missing credential or a mistyped code can't kill the kernel. A
    `recruitradar.session` file is written on success so later runs are silent.
    """
    global client
    if not TELETHON_OK:
        return None
    if not CREDENTIALS_OK:
        print("Skipping login: credentials incomplete (re-run section 1 to add them).")
        return None

    client = TelegramClient(SESSION_NAME, API_ID_INT, API_HASH)
    try:
        await client.connect()
    except Exception as e:
        print(f"Could not connect to Telegram: {e}")
        client = None
        return None

    # Already have a valid session? Done.
    if await client.is_user_authorized():
        me = await client.get_me()
        print("Already authorized as:", me.username or me.first_name, "| id:", me.id)
        return client

    # Ask Telegram to send the login code to the user's app.
    try:
        await client.send_code_request(PHONE)
    except PhoneNumberInvalidError:
        print("Phone number rejected by Telegram. Fix TELEGRAM_PHONE (intl format) and re-run.")
        return None
    except FloodWaitError as e:
        print(f"Rate-limited before login — wait {e.seconds}s and re-run this cell.")
        return None
    except Exception as e:
        print(f"Could not request a login code: {e}")
        return None

    # Up to 3 attempts at the login code.
    for attempt in range(1, 4):
        code_ = _ask("Enter the login code Telegram sent to your app: ")
        if not code_:
            print("No code entered — login aborted (kernel stays alive). Re-run to retry.")
            return None
        try:
            await client.sign_in(PHONE, code_)
            break
        except SessionPasswordNeededError:
            # Two-factor auth is enabled — ask for the 2FA password (up to 3 tries).
            for _ in range(3):
                pw = _ask("Two-step (2FA) password: ", secret=True)
                if not pw:
                    print("No 2FA password entered — login aborted.")
                    return None
                try:
                    await client.sign_in(password=pw)
                    break
                except Exception:
                    print("  Wrong 2FA password — try again.")
            break
        except (PhoneCodeInvalidError, PhoneCodeExpiredError):
            print(f"  Invalid/expired code ({attempt}/3). "
                  "Request a fresh one from Telegram if it expired.")
        except Exception as e:
            print(f"  Login error: {e}")
            return None

    if await client.is_user_authorized():
        me = await client.get_me()
        print("Logged in as:", me.username or me.first_name, "| id:", me.id)
        return client

    print("Not authorized — live collection stays disabled. Re-run this cell to retry.")
    return None


client = await telegram_login()


## 3. Channel registry — the hunting grounds

Recruitment posts hide among ordinary offers, so we monitor the everyday
channels where Israelis look for **money, work and second-hand deals**, in all
three relevant languages (Hebrew / Russian / English):

| category | what it covers |
|---|---|
| `jobs_il` | jobs inside Israel (incl. Russian-language boards) |
| `jobs_abroad` | jobs abroad / relocation offers |
| `jobs_it` | hi-tech / IT vacancies |
| `freelance` | freelance gigs, micro-tasks |
| `translation` | translation / language jobs |
| `field_work` | physical field jobs (drivers, couriers, manual work) |
| `photo_video` | photography / video shooting gigs |
| `help_offers` | "offering/seeking help" community groups |
| `trips_travel` | trips, travel-together, meetups |
| `yad2` / `marketplace` / `furniture` | classified boards, buy/sell, furniture |
| `crypto_exchange` | crypto exchanger offers & jobs |
| `rides` | טרמפים / rides |

**News channels are intentionally absent** — they only add noise.

### Adding more channels without editing code
Put extra channels in **`channels_extra.txt`** (one per line, `username, category`).
The cell below loads and de-duplicates them automatically — drop in the file
you're preparing and re-run.

Find candidates on: https://telegramsearchengine.com/ · https://tgstat.com/ ·
https://www.telemetryapp.io/ · https://nicegram.app/hub/search · https://www.tgdb.org/
(search `דרושים`, `עבודה`, `לוח`, `טרמפים`, `работа в Израиле`, `подработка`,
`барахолка`, `обменник`, `jobs israel`, `freelance` …)


In [ ]:
# (channel_username_or_id, category) — PUBLIC channels only. News removed.
#
# ✅ = resolved & collected in a previous live run on this account
# 🔎 = found on a public directory (tgstat/telemetr/search), not yet collected
# ❓ = plausible-name seed, NOT verified — the collector skips it if dead
CHANNELS = [
    # --- jobs in Israel ------------------------------------------------------
    ("israjobs",         "jobs_it"),        # ✅ hi-tech vacancies (RU/EN)
    ("jobs_in_israel",   "jobs_il"),        # ✅ blue-collar jobs (RU)
    ("ConnectJLMJobs",   "jobs_il"),        # ✅ resolves; Olim jobs Jerusalem (EN)
    ("BROOTTO_Jobs",     "jobs_il"),        # 🔎 Работа в Израиле
    ("rabotaisraeli",    "jobs_il"),        # 🔎 Работа Израиль / עבודה בישראל
    ("rabotacoil",       "jobs_il"),        # 🔎 Rabota.co.il דרושים ברוסית

    # --- jobs abroad / relocation -------------------------------------------
    ("rabota_za_granicey",  "jobs_abroad"), # ❓
    ("jobs_abroad_il",      "jobs_abroad"), # ❓

    # --- freelance / micro-gigs / translation / photo-video ------------------
    ("freelance_il",        "freelance"),   # ❓
    ("freelancim",          "freelance"),   # ❓ Hebrew freelancers board
    ("perevodchiki_rabota", "translation"), # ❓ RU translation gigs
    ("tsalamim_il",         "photo_video"), # ❓ Hebrew photographers board

    # --- field work / couriers ----------------------------------------------
    ("shlichuyot_il",       "field_work"),  # ❓ couriers/deliveries
    ("avoda_baregel",       "field_work"),  # ❓

    # --- help / community / trips -------------------------------------------
    ("ezra_hadadit",        "help_offers"), # ❓ mutual-help groups
    ("tiyulim_israel",      "trips_travel"),# ❓ trips & travel together
    ("trempim_israel",      "rides"),       # ✅ rides (low volume)

    # --- marketplaces / yad2 / furniture -------------------------------------
    ("izrail_avito",        "marketplace"), # 🔎 ИЗРАИЛЬ | объявления барахолка
    ("yad2_il",             "yad2"),        # ❓
    ("kupi_proday_israel",  "marketplace"), # ❓ RU buy/sell
    ("rahitim_yad2",        "furniture"),   # ❓ furniture second-hand

    # --- crypto exchangers ----------------------------------------------------
    ("crypto_exchange_il",  "crypto_exchange"), # ❓
    ("obmen_valut_israel",  "crypto_exchange"), # ❓ RU обменник
]

# ── Extra channels file (no code edits needed) ──────────────────────────────
# Format per line:  username, category      (lines starting with # are ignored)
CHANNELS_FILE = Path("channels_extra.txt")
VALID_CATEGORIES = {
    "jobs_il", "jobs_abroad", "jobs_it", "freelance", "translation",
    "field_work", "photo_video", "help_offers", "trips_travel",
    "yad2", "marketplace", "furniture", "crypto_exchange", "rides", "tutoring",
}
if CHANNELS_FILE.exists():
    known = {u for u, _ in CHANNELS}
    added = 0
    for line in CHANNELS_FILE.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip().lstrip("@") for p in line.replace("\t", ",").split(",")]
        if not parts or not parts[0]:
            continue
        uname = parts[0]
        cat = parts[1] if len(parts) > 1 and parts[1] else "uncategorized"
        if cat not in VALID_CATEGORIES and cat != "uncategorized":
            print(f"  note: '{uname}' has unknown category '{cat}' (kept anyway)")
        if uname not in known:
            CHANNELS.append((uname, cat))
            known.add(uname)
            added += 1
    print(f"Loaded {added} extra channel(s) from {CHANNELS_FILE}.")
else:
    print(f"(no {CHANNELS_FILE} found — drop one in later and re-run this cell)")

# How far back to collect, and a per-channel cap to keep runs cheap.
DAYS_BACK        = 180
MAX_PER_CHANNEL  = 500       # per-channel cap; None = everything in the window
SLEEP_BETWEEN    = 0.05      # seconds between messages

print(f"{len(CHANNELS)} channels registered "
      f"({len({c for _, c in CHANNELS})} categories).")


## 4. Storage — SQLite schema + raw-JSON archive

In [ ]:
def init_db(path=DB_PATH):
    conn = sqlite3.connect(path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            channel         TEXT    NOT NULL,
            category        TEXT,
            msg_id          INTEGER NOT NULL,
            date            TEXT,                 -- ISO8601 UTC
            sender_hash     TEXT,                 -- pseudonymized sender
            text            TEXT,
            has_media       INTEGER,
            forwards        INTEGER,
            views           INTEGER,
            replies         INTEGER,
            is_forwarded    INTEGER,
            collected_at    TEXT,
            PRIMARY KEY (channel, msg_id)
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_msg_date    ON messages(date)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_msg_sender  ON messages(sender_hash)")
    conn.commit()
    return conn

conn = init_db()
print("SQLite ready at", DB_PATH)


In [ ]:
def save_message(conn, channel, category, m):
    """Insert one Telethon message into SQLite (idempotent via PRIMARY KEY)."""
    replies = m.replies.replies if getattr(m, "replies", None) else 0
    conn.execute(
        """INSERT OR IGNORE INTO messages
           (channel, category, msg_id, date, sender_hash, text, has_media,
            forwards, views, replies, is_forwarded, collected_at)
           VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
        (
            channel, category, m.id,
            m.date.astimezone(timezone.utc).isoformat() if m.date else None,
            hash_user_id(m.sender_id),
            m.text or "",
            1 if m.media is not None else 0,
            m.forwards or 0,
            m.views or 0,
            replies,
            1 if m.fwd_from is not None else 0,
            datetime.now(timezone.utc).isoformat(),
        ),
    )

def raw_record(channel, category, m):
    """Provenance archive — keep the real id here LOCALLY only (gitignored)."""
    return {
        "channel": channel, "category": category, "msg_id": m.id,
        "date": m.date.isoformat() if m.date else None,
        "sender_id": m.sender_id, "text": m.text,
        "has_media": m.media is not None, "forwards": m.forwards,
        "views": m.views, "is_forwarded": m.fwd_from is not None,
    }


## 5–6. Rate-limited bulk collection

Iterates each channel back `DAYS_BACK` days, writes to SQLite (pseudonymized)
and to a per-channel raw JSONL archive. Handles `FloodWaitError` by sleeping
exactly as long as Telegram asks, and skips private/missing channels.


In [ ]:
# ── Section 6: Rate-limited bulk collection ──────────────────────────────
# Key changes vs original:
#   • SLEEP_BETWEEN lowered to 0.05 s  (Telegram allows ~20 req/s for reads)
#   • MAX_PER_CHANNEL hard-caps each channel (default 500 for first run)
#   • Per-channel wall-clock timeout (CHANNEL_TIMEOUT_SEC) aborts a runaway
#     channel and moves on — set to 0 to disable.
#   • Batch-commit every 500 rows instead of every 200 (fewer fsync stalls)
#   • Progress counter printed every 100 messages

import time

SLEEP_BETWEEN       = 0.05   # seconds between messages (safe for public channels)
MAX_PER_CHANNEL     = 500    # hard cap per channel; set None for "all in window"
CHANNEL_TIMEOUT_SEC = 120    # max seconds per channel; 0 = no timeout


async def collect_channel(conn, channel, category,
                           days_back=DAYS_BACK, max_msgs=MAX_PER_CHANNEL):
    cutoff   = datetime.now(timezone.utc) - timedelta(days=days_back)
    raw_path = RAW_DIR / f"{str(channel).replace('/', '_')}.jsonl"
    n        = 0
    t_start  = time.monotonic()
    try:
        with open(raw_path, 'a', encoding='utf-8') as raw_f:
            async for m in client.iter_messages(channel, limit=max_msgs or None):
                # date cutoff
                if m.date and m.date.astimezone(timezone.utc) < cutoff:
                    break
                # wall-clock timeout guard
                if CHANNEL_TIMEOUT_SEC and (time.monotonic() - t_start) > CHANNEL_TIMEOUT_SEC:
                    print(f"  timeout after {CHANNEL_TIMEOUT_SEC}s — moving on")
                    break
                save_message(conn, str(channel), category, m)
                raw_f.write(json.dumps(raw_record(str(channel), category, m),
                                       ensure_ascii=False) + '\n')
                n += 1
                if n % 100 == 0:
                    print(f"    {n} messages collected...", flush=True)
                if n % 500 == 0:
                    conn.commit()
                if max_msgs and n >= max_msgs:
                    break
                await asyncio.sleep(SLEEP_BETWEEN)
    except FloodWaitError as e:
        print(f"  FloodWait on {channel}: sleeping {e.seconds}s")
        await asyncio.sleep(e.seconds + 1)
    except (ChannelPrivateError, UsernameNotOccupiedError, ValueError) as e:
        print(f"  skip {channel}: {type(e).__name__}")
    conn.commit()
    elapsed = time.monotonic() - t_start
    return n, elapsed


async def collect_all():
    totals = {}
    for channel, category in CHANNELS:
        print(f"Collecting {channel} ({category}) ...", flush=True)
        try:
            n, elapsed = await collect_channel(conn, channel, category)
            print(f"  -> {n} messages in {elapsed:.0f}s")
            totals[channel] = n
        except Exception as e:
            print(f"  ERROR {channel}: {e}")
            totals[channel] = 0
    return totals


if client is not None:
    totals = await collect_all()
    print("Done:", totals, "| total messages:", sum(totals.values()))
else:
    totals = {}
    print("No Telegram client — skipping live collection. "
          "Sections 7–10 will use demo / previously-collected data.")


## 7. Plain-text snapshot database (`data/txt/`)

One **`.txt` file per channel**, rewritten on every run, sorted by message id —
one line per message (`msg_id <TAB> date <TAB> sender_hash <TAB> text`).
Because the files are stable and sorted, you can see exactly **what changed in
each group over time** with any diff tool:

```bash
# example: keep dated copies and compare runs
cp -r data/txt data/txt_2026-06-11
diff data/txt_2026-06-11/israjobs.txt data/txt/israjobs.txt
```

A `_runs_log.txt` records each run's per-channel counts, so growth over time is
visible at a glance even without diffing.


In [ ]:
def _flat(text):
    """One message = one line: collapse newlines/tabs so the txt stays diffable."""
    return (text or "").replace("\\", "\\\\").replace("\t", "\\t").replace("\r", "").replace("\n", "\\n")


def export_txt_snapshots(conn):
    """Rewrite data/txt/<channel>.txt from SQLite (sorted, stable → diffable),
    and append a per-run summary line to data/txt/_runs_log.txt."""
    channels = [r[0] for r in conn.execute(
        "SELECT DISTINCT channel FROM messages ORDER BY channel")]
    counts = {}
    for ch in channels:
        rows = conn.execute(
            """SELECT msg_id, date, sender_hash, text FROM messages
               WHERE channel = ? ORDER BY msg_id""", (ch,)).fetchall()
        path = TXT_DIR / f"{str(ch).replace('/', '_')}.txt"
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"# channel: {ch}\n# messages: {len(rows)}\n"
                    f"# format: msg_id<TAB>date<TAB>sender_hash<TAB>text (newlines as \\n)\n")
            for msg_id, date, sender, text in rows:
                f.write(f"{msg_id}\t{date or ''}\t{sender or ''}\t{_flat(text)}\n")
        counts[ch] = len(rows)

    log_line = (datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC") + "  " +
                "  ".join(f"{ch}={n}" for ch, n in sorted(counts.items())))
    with open(TXT_DIR / "_runs_log.txt", "a", encoding="utf-8") as f:
        f.write(log_line + "\n")
    return counts


snapshot_counts = export_txt_snapshots(conn)
print(f"Wrote {len(snapshot_counts)} channel snapshot(s) to {TXT_DIR}/:")
for ch, n in sorted(snapshot_counts.items(), key=lambda x: -x[1]):
    print(f"  {ch:<28} {n:>6} messages")


## 8. EDA

Everything below reads from SQLite, so you can iterate offline. If you haven't
collected yet, the next cell seeds a tiny **synthetic** sample so the whole
pipeline runs end-to-end as a demo (clearly marked, channel `__demo__`).


In [ ]:
def seed_demo_if_empty(conn):
    cur = conn.execute("SELECT COUNT(*) FROM messages")
    if cur.fetchone()[0] > 0:
        return
    print("No real data yet — seeding synthetic demo rows (channel '__demo__').")
    demo = [
        "דרוש נהג למשמרות בוקר באזור המרכז, שכר שעתי הוגן.",
        "מחפשים מתאם.ת לוגיסטיקה, ניסיון יתרון.",
        "משימה קטנה וקלה, תשלום מיידי במזומן/קריפטו. פנו בפרטי לטלגרם מאובטח.",
        "צריך מישהו לצלם בניין ולתלות כמה כרזות, כסף קל ומהיר, אנונימי לחלוטין.",
        "טרמפ מתל אביב לחיפה מחר ב-8 בבוקר.",
        "מורה פרטי למתמטיקה לכיתה ט', אזור ירושלים.",
        "עבודה מהבית, רווח גבוה ביום, תשלום בביטקוין, צרו קשר בסיגנל.",
    ]
    now = datetime.now(timezone.utc)
    for i, t in enumerate(demo):
        conn.execute(
            """INSERT OR IGNORE INTO messages
               (channel, category, msg_id, date, sender_hash, text, has_media,
                forwards, views, replies, is_forwarded, collected_at)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
            ("__demo__", "demo", i,
             (now - timedelta(days=i)).isoformat(),
             hash_user_id(900000 + i), t, 0, 0, 10 * i, 0, 0,
             now.isoformat()))
    conn.commit()

seed_demo_if_empty(conn)
df = pd.read_sql_query("SELECT * FROM messages", conn, parse_dates=["date"])
print(f"{len(df)} messages across {df.channel.nunique()} channels.")
df.head()


In [ ]:
# Volume by channel & category
print(df.groupby("category").size().sort_values(ascending=False))
print()
print("unique senders:", df.sender_hash.nunique())
print("messages with media:", int(df.has_media.sum()))
print("forwarded messages:", int(df.is_forwarded.sum()))


In [ ]:
# Posting volume over time (simple visual sanity check)
import matplotlib.pyplot as plt
if df["date"].notna().any():
    ts = df.dropna(subset=["date"]).set_index("date").resample("D").size()
    ax = ts.plot(figsize=(10, 3), title="Messages per day")
    ax.set_ylabel("count"); plt.tight_layout(); plt.show()


## 9. Weak supervision — trilingual rule lexicon → noisy labels

We encode the **documented** recruitment patterns as keyword/regex rules in
**Hebrew, Russian and English**. Each rule contributes weight to a `rule_score`.
This is intentionally noisy — a positive *seed set* to bootstrap a model, **not**
a verdict.

This is only **half** the detector. Keyword rules catch posts that *look like*
the documented playbook, but a real recruitment post may be phrased completely
differently. Section 11 adds an **appearance-anomaly** model that flags posts
which simply don't look like the rest of *their own group* — even with zero
keyword hits.

Signal categories: `easy_money`, `crypto`, `tasking` (photograph/hang/spray/
deliver), `opsec` (move-to-private/delete/discreet), `target_sites`,
`recruitment_framing`, `urgency`, `pretext` (drone/courier/PI/dating).


In [ ]:
import re

# EXCLUDED_CATEGORIES: channels in these categories are never keyword-scored.
EXCLUDED_CATEGORIES = {"news"}

# ── Trilingual lexicon (Hebrew + Russian + English) ─────────────────────────
# Patterns derived from publicly reported recruitment playbook. Intentionally
# noisy. Russian matters: many Israeli jobs/marketplace channels are RU-language.
RULES = {
    # Money offers that are suspiciously easy / immediate / no-questions
    "easy_money": [
        # Hebrew
        r"כסף\s*קל", r"כסף\s*מהיר", r"כסף\s*בקלות", r"רווח\s*(מהיר|גבוה|קל)",
        r"תשלום\s*מיידי", r"\bבלי\s*ניסיון\b", r"ללא\s*ניסיון",
        r"סכום\s*גבוה", r"מאות\s*(דולר|שקל|אירו)", r"אלפי\s*(שקל|דולר)",
        r"הכנסה\s*(נוספת|קלה|מהירה)", r"כסף\s*מהיר\s*וקל",
        # Russian
        r"лёгк\w*\s*деньг", r"быстр\w*\s*деньг", r"лёгкий\s*заработок",
        r"высок\w*\s*(оплат|доход|заработок)", r"без\s*опыта",
        r"хорошо\s*плат", r"оплата\s*сразу", r"мгновенн\w*\s*оплат",
        r"\d{3,}\s*(\$|долл|шекел|₪|евро)\s*(в\s*день|за\s*час)?",
        # English
        r"\beasy\s*(money|cash)\b", r"\bquick\s*(money|cash)\b", r"\bfast\s*cash\b",
        r"\bearn\s*(fast|easy|big|extra)\b", r"\bgood\s*(money|pay)\b",
        r"\bno\s*experience\b", r"\bpaid\s*(immediately|instantly|same\s*day)\b",
    ],
    # Crypto payment — strong signal when combined with other categories
    "crypto": [
        r"קריפטו", r"ביטקוין", r"את'?ריום", r"אתריום", r"מטבע\s*דיגיטלי",
        r"ארנק\s*(דיגיטלי|קריפטו)", r"תשלום\s*ב(קריפטו|ביטקוין|מטבע)",
        r"крипт", r"биткоин", r"эфири", r"кошел[её]к", r"оплата\s*в\s*крипт",
        r"\bbtc\b", r"\busdt\b", r"\busdc\b", r"\beth\b", r"\bton\b",
        r"\bbitcoin\b", r"\btether\b", r"\bbinance\b", r"\bcrypto\s*wallet\b",
        r"\bpay(ment)?\s*in\s*crypto\b",
    ],
    # Physical tasking: photograph, hang posters, spray, deliver/collect package
    "tasking": [
        # Hebrew
        r"משימה\s*(קטנה|קלה|פשוטה|זריזה)", r"מטלה\s*קטנה",
        r"לצלם\s+(בניין|מתקן|אתר|מקום|אזור|צומת|כביש|נקוד)",
        r"לתלות\s*(כרזות|שלטים|פלייר|כרזה|מודעות)",
        r"להעביר\s*חבילה", r"לאסוף\s*(חבילה|מעטפה|חפץ)", r"להניח\s*(חבילה|מעטפה|חפץ)",
        r"ריסוס", r"גרפיטי", r"לרסס", r"לכתוב\s*על\s*קיר",
        r"להדביק\s*(מדבקות|מדבקה|כרזות)", r"להצית", r"להבעיר", r"לשרוף", r"הצתה",
        # Russian
        r"сфотографир", r"сделать\s*фото\s*(объект|здани|мест)",
        r"расклеить\s*(листовк|плакат|наклейк)", r"граффити", r"распыл",
        r"забрать\s*(посылк|пакет|конверт)", r"оставить\s*(посылк|пакет|конверт)",
        r"передать\s*(посылк|пакет)", r"поджечь", r"поджог",
        r"маленьк\w*\s*задани", r"простое\s*задани",
        # English
        r"\bspray\s*paint\b", r"\bhang\s*(posters|signs|flyers)\b",
        r"\bdrop\s*(off\s*)?(a\s*)?package\b", r"\bdeliver\s*(a\s*)?package\b",
        r"\bpick\s*up\s*(a\s*)?(package|envelope)\b", r"\bphotograph\s*(the|a)\s*\w+",
        r"\bsmall\s*task\b", r"\bsimple\s*task\b",
    ],
    # OPSEC / move-to-private — must be about MOVING communication, not just a link
    "opsec": [
        # Hebrew
        r"פנו?\s+בפרטי", r"שלחו\s+בפרטי", r"נמשיך\s+ב(סיגנל|וואטסאפ|פרטי|אפליקציה)",
        r"עבור(ו)?\s+ל(סיגנל|וואטסאפ|אפליקציה\s*מאובטחת)",
        r"צרו\s+קשר\s+ב(סיגנל|וואטסאפ)", r"דברו\s+איתי\s+ב(סיגנל|וואטסאפ|פרטי)",
        r"אפליקציה\s*מאובטחת", r"מחק\s*אחרי", r"מחק\s*את\s*ההודעה",
        r"נמחק\s*אוטומטי", r"דיסקרטי", r"בדיסקרטיות", r"שמור\s*בסוד",
        r"אל\s*תספר\s*לאף\s*אחד", r"בלי\s*שאלות", r"לא\s*שואלים\s*שאלות",
        # Russian
        r"пишите\s*в\s*(личк|лс)", r"в\s*личные\s*сообщени", r"переход\w*\s*в\s*(сигнал|signal|вотсап|whatsapp)",
        r"продолжим\s*в\s*(сигнал|signal|личк)", r"удали\w*\s*после",
        r"анонимн", r"конфиденциальн", r"никому\s*не\s*говори", r"без\s*лишних\s*вопрос",
        # English
        r"\bdm\s*me\b", r"\bpm\s*me\b", r"\bmessage\s*me\s*privately\b",
        r"\bmove\s*to\s*(signal|whatsapp|private)\b", r"\bcontinue\s*(on|in)\s*signal\b",
        r"\bno\s*questions\b", r"\bdiscreet\b", r"\bdelete\s*after\b", r"\banonymous\b",
    ],
    # Sensitive/military sites — only meaningful when paired with TASKING or OPSEC
    "target_sites": [
        r"בסיס\s*צבאי", r"מתקן\s*(צבאי|ביטחוני|רגיש)", r"תחנת\s*כוח",
        r"נמל\s*תעופה", r"שדה\s*תעופה", r"מצלמ(ה|ות)\s*אבטחה\s*(ליד|על|ב)",
        r"כיפת\s*ברזל", r"מערך\s*הגנה", r"תשתית\s*(קריטית|חיונית|לאומית)",
        r"מחסום", r"שגרירות", r"תחנת\s*משטרה",
        r"военн\w*\s*баз", r"электростанц", r"аэропорт", r"посольств",
        r"критическ\w*\s*инфраструктур", r"блокпост",
        r"\bmilitary\s*base\b", r"\biron\s*dome\b", r"\bpower\s*(plant|station)\b",
        r"\bcheckpoint\b", r"\bembassy\b",
    ],
    # Recruitment framing: looking for people for vague paid work
    "recruitment_framing": [
        r"מחפש(ים|ת)?\s+אנשים\s+(ל|עבור)", r"דרוש(ים|ה)?\s+אנשים\s+(ל|עבור)",
        r"עבודה\s*מהטלפון", r"להרוויח\s*מהטלפון", r"גם\s+בני\s+נוער",
        r"מתאים\s+גם\s+לקטינים", r"מקומות\s*אחרונים",
        r"ищем\s*(людей|человек)\s*для", r"требуются\s*люди\s*для",
        r"работа\s*с\s*телефон", r"подработк\w*\s*для\s*(студент|подрост|молод)",
        r"можно\s*без\s*опыта", r"удал[её]нн\w*\s*работ",
        r"\bwork\s*from\s*(home|phone)\b", r"\bside\s*gig\b",
        r"\blooking\s*for\s*people\s*(to|for)\b", r"\bremote\s*(work|job)\b",
    ],
    # Urgency words — only scored as a booster (low weight), never standalone
    "urgency": [
        r"דחוף", r"מיידי", r"זמן\s*מוגבל", r"רק\s*היום",
        r"срочно", r"сегодня\s*только", r"ограниченн\w*\s*врем",
        r"\burgent\b", r"\basap\b", r"\btoday\s*only\b",
    ],
    # Deceptive pretexts documented in indictments
    "pretext": [
        r"צילומי\s*דרון", r"רחפן", r"דרון", r"שליח", r"שירותי\s*שליחות", r"חוקר\s*פרטי",
        r"дрон", r"квадрокоптер", r"курьер", r"частн\w*\s*детектив",
        r"\bdrone\b", r"\bcourier\b", r"\bprivate\s*investigator\b",
    ],
}
RULE_WEIGHTS = {
    "easy_money": 1.0, "crypto": 2.0, "tasking": 2.5, "opsec": 2.0,
    "target_sites": 1.5, "recruitment_framing": 1.0, "urgency": 0.3, "pretext": 1.0,
}
COMPILED = {k: [re.compile(p, re.IGNORECASE) for p in v] for k, v in RULES.items()}


def apply_rules(text):
    text = text or ""
    hits, score = {}, 0.0
    for cat, patterns in COMPILED.items():
        matched = [p.pattern for p in patterns if p.search(text)]
        if matched:
            hits[cat] = matched
            score += RULE_WEIGHTS[cat]
    return score, hits


# Apply to corpus — skip excluded categories entirely
mask = ~df["category"].isin(EXCLUDED_CATEGORIES)
res = df.loc[mask, "text"].apply(apply_rules)
df["rule_score"] = 0.0
df["rule_hits"] = [[] for _ in range(len(df))]
df.loc[mask, "rule_score"] = res.apply(lambda r: r[0])
df.loc[mask, "rule_hits"]  = res.apply(lambda r: list(r[1].keys()))
df["n_rule_cats"] = df["rule_hits"].apply(len)

print(f"Lexicon: {sum(len(v) for v in RULES.values())} patterns across {len(RULES)} "
      f"categories (Hebrew + Russian + English).")
print(f"Excluded categories (not scored): {EXCLUDED_CATEGORIES}")
print(f"Scored messages: {int(mask.sum())} / {len(df)} total")
print(df[df.rule_score > 0][["rule_score", "n_rule_cats"]].describe())
df.sort_values("rule_score", ascending=False)[["channel", "rule_score", "rule_hits", "text"]].head(10)


## 10. Behavioral features + suspicion score

Recruitment posts often pair suspicious **text** with suspicious **behavior**:
a sender active in few channels, short-lived bursts, low engagement. We combine
the rule score with light behavioral features into a single ranked score.

> The weights are heuristic. Once you have labels, replace this with a trained
> classifier (XGBoost on these features, AlephBERT on the text).


In [ ]:
# Sender-level behavioral aggregates
sender_stats = df.groupby("sender_hash").agg(
    n_msgs     = ("msg_id", "count"),
    n_channels = ("channel", "nunique"),
    avg_views  = ("views", "mean"),
    max_rule   = ("rule_score", "max"),
).reset_index()

sender_stats["multi_channel_low_engagement"] = (
    (sender_stats["n_channels"] >= 2)
    & (sender_stats["avg_views"] < df["views"].median())
).astype(int)

df = df.merge(
    sender_stats[["sender_hash", "n_channels", "multi_channel_low_engagement"]],
    on="sender_hash", how="left")


def suspicion_score(row):
    # Skip news / control channels entirely
    if row.get("category") in EXCLUDED_CATEGORIES:
        return 0.0

    hits = set(row["rule_hits"])
    s    = row["rule_score"]

    # 'target_sites' alone in a news context is just reporting — only boost
    # when it co-occurs with tasking or opsec (someone being SENT somewhere).
    if "target_sites" in hits and not hits & {"tasking", "opsec", "easy_money"}:
        s -= RULE_WEIGHTS["target_sites"]   # cancel the target_sites contribution

    # Multiple independent signal categories → stronger
    if row["n_rule_cats"] >= 2:
        s += 1.0
    if row["n_rule_cats"] >= 3:
        s += 1.0   # triple-hit is much more telling

    if row["multi_channel_low_engagement"]:
        s += 0.5

    # Original (non-forwarded) posts are more telling than reposts
    if row.get("is_forwarded"):
        s -= 0.3

    return max(s, 0.0)


df["suspicion"] = df.apply(suspicion_score, axis=1)
ranked = df.sort_values("suspicion", ascending=False)
ranked[["channel", "suspicion", "rule_hits", "n_channels", "text"]].head(15)


In [ ]:
# The review queue: highest-suspicion posts for MANUAL verification.
THRESHOLD = 2.0
queue = ranked[ranked["suspicion"] >= THRESHOLD].copy()
print(f"{len(queue)} posts above suspicion threshold {THRESHOLD} — review these by hand.")
queue[["channel", "category", "date", "suspicion", "rule_hits", "text"]].head(30)


## 11. Appearance anomalies — what looks *odd for this group*

The keyword rules only catch what we already know to look for. A real
recruitment post may use none of those words yet still **stand out from the
rest of its group**: a different language, an unusual length, a wallet address,
a "DM me" with no normal job details, contact info where the group never puts
any.

So for **each channel separately** we build a numeric fingerprint of every
message (length, language mix, links, phone/wallet patterns, emoji, mentions,
media, engagement) and fit an **Isolation Forest** — an unsupervised model that
learns "normal for this group" and scores how far each post deviates. No labels
needed. Output: `appearance_anomaly` ∈ [0, 1] (higher = stranger).

We deliberately fit **per channel** because "normal" differs wildly: a Russian
blue-collar board, a Hebrew rides group and an English hi-tech channel each have
their own baseline.


In [ ]:
from sklearn.ensemble import IsolationForest

# ── Per-message appearance features ─────────────────────────────────────────
URL_RE     = re.compile(r"https?://|t\.me/|www\.")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
PHONE_RE   = re.compile(r"(?:\+?\d[\d\-\s]{7,}\d)")
WALLET_RE  = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34}|T[A-Za-z1-9]{33})\b")
EMOJI_RE   = re.compile("[\U0001F000-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]")
HEB_RE     = re.compile(r"[֐-׿]")
CYR_RE     = re.compile(r"[Ѐ-ӿ]")
LAT_RE     = re.compile(r"[A-Za-z]")
TGUSER_RE  = re.compile(r"tg://user")


def appearance_features(text, has_media, views, forwards, replies, is_forwarded):
    t = text or ""
    L = max(len(t), 1)
    letters = HEB_RE.findall(t), CYR_RE.findall(t), LAT_RE.findall(t)
    nl = sum(len(x) for x in letters) or 1
    return {
        "len":          len(t),
        "n_words":      len(t.split()),
        "avg_word_len": (sum(len(w) for w in t.split()) / max(len(t.split()), 1)),
        "digit_ratio":  sum(c.isdigit() for c in t) / L,
        "upper_ratio":  sum(c.isupper() for c in t) / L,
        "n_urls":       len(URL_RE.findall(t)),
        "n_mentions":   len(MENTION_RE.findall(t)),
        "n_hashtags":   len(HASHTAG_RE.findall(t)),
        "n_emoji":      len(EMOJI_RE.findall(t)),
        "has_phone":    int(bool(PHONE_RE.search(t))),
        "has_wallet":   int(bool(WALLET_RE.search(t))),
        "has_tguser":   int(bool(TGUSER_RE.search(t))),
        "heb_ratio":    len(letters[0]) / nl,
        "cyr_ratio":    len(letters[1]) / nl,
        "lat_ratio":    len(letters[2]) / nl,
        "has_media":    int(bool(has_media)),
        "views":        float(views or 0),
        "forwards":     float(forwards or 0),
        "replies":      float(replies or 0),
        "is_forwarded": int(bool(is_forwarded)),
    }


feat_rows = df.apply(lambda r: appearance_features(
    r["text"], r["has_media"], r["views"], r.get("forwards", 0),
    r.get("replies", 0), r["is_forwarded"]), axis=1)
FEATS = pd.DataFrame(list(feat_rows), index=df.index)
FEATURE_COLS = list(FEATS.columns)
df[FEATURE_COLS] = FEATS

# ── Fit one Isolation Forest PER channel ────────────────────────────────────
MIN_FOR_MODEL = 30      # channels smaller than this get a fallback score
df["appearance_anomaly"] = 0.0

for ch, idx in df.groupby("channel").groups.items():
    if df.loc[idx, "category"].iloc[0] in EXCLUDED_CATEGORIES:
        continue
    sub = FEATS.loc[idx]
    X = sub.fillna(0.0).values
    if len(idx) >= MIN_FOR_MODEL and np.ptp(X, axis=0).sum() > 0:
        iso = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
        iso.fit(X)
        raw = -iso.score_samples(X)                      # higher = more anomalous
        rng = raw.max() - raw.min()
        norm = (raw - raw.min()) / rng if rng > 0 else np.zeros_like(raw)
    else:
        # too few messages to model — fall back to simple z-score of length+links
        z = sub[["len", "n_urls", "has_wallet", "has_phone"]].fillna(0).sum(axis=1)
        norm = ((z - z.min()) / (z.max() - z.min())).values if z.max() > z.min() else np.zeros(len(idx))
    df.loc[idx, "appearance_anomaly"] = norm

scored = df[~df["category"].isin(EXCLUDED_CATEGORIES)]
print(f"Fitted appearance models over {scored['channel'].nunique()} channels, "
      f"{len(scored)} messages.")
print("\nMost anomalous-looking posts per their own group (top 12):")
cols = ["channel", "appearance_anomaly", "rule_score", "rule_hits", "text"]
top_app = scored.sort_values("appearance_anomaly", ascending=False)[cols].head(12).copy()
top_app["text"] = top_app["text"].str.slice(0, 110)
top_app


In [ ]:
# ── Combine the two detectors ───────────────────────────────────────────────
# Normalize keyword suspicion to [0,1] across the scored corpus, then blend with
# the per-group appearance anomaly. Two independent reasons to look = a stronger
# lead than either alone.
W_KEYWORD, W_APPEARANCE = 0.6, 0.4

s = df["suspicion"].fillna(0.0)
df["suspicion_norm"] = (s / s.max()) if s.max() > 0 else 0.0
df["flag_score"] = (W_KEYWORD * df["suspicion_norm"]
                    + W_APPEARANCE * df["appearance_anomaly"]).round(4)
# Excluded categories contribute nothing
df.loc[df["category"].isin(EXCLUDED_CATEGORIES), "flag_score"] = 0.0

# A post is worth a human's time if EITHER detector lights up.
KEYWORD_MIN, APPEARANCE_MIN = 2.0, 0.75
df["flag_reason"] = np.where(
    (df["suspicion"] >= KEYWORD_MIN) & (df["appearance_anomaly"] >= APPEARANCE_MIN), "both",
    np.where(df["suspicion"] >= KEYWORD_MIN, "keyword",
    np.where(df["appearance_anomaly"] >= APPEARANCE_MIN, "appearance", "")))

flagged = df[df["flag_reason"] != ""].sort_values("flag_score", ascending=False)
print(f"{len(flagged)} flagged posts "
      f"(keyword-only: {(flagged.flag_reason=='keyword').sum()}, "
      f"appearance-only: {(flagged.flag_reason=='appearance').sum()}, "
      f"both: {(flagged.flag_reason=='both').sum()})")
view = flagged[["channel", "flag_reason", "flag_score", "suspicion",
                "appearance_anomaly", "rule_hits", "text"]].head(20).copy()
view["text"] = view["text"].str.slice(0, 100)
view


## 12. Dashboard

Builds a single self-contained **`exports/dashboard.html`** (open in any
browser) with two ranked tabs:

1. **Keyword anomalies** — posts that match the trilingual recruitment lexicon.
2. **Appearance anomalies** — posts that look odd vs. the rest of *their own
   group* (Isolation Forest), even with no keyword hit.

Plus a per-channel summary (volume, flag rate, dominant language) so you can see
which groups are worth a closer look. Sender ids are already pseudonymized; no
raw PII is written.


In [ ]:
import html as _html


def _rows(frame, score_col, n=200):
    out = []
    for _, r in frame.head(n).iterrows():
        hits = ", ".join(r["rule_hits"]) if isinstance(r["rule_hits"], list) else ""
        txt = _html.escape((r["text"] or "")[:600])
        date = str(r["date"])[:16]
        out.append(
            f"<tr><td>{r[score_col]:.3f}</td><td>{_html.escape(str(r['channel']))}</td>"
            f"<td>{_html.escape(str(r['category']))}</td><td>{date}</td>"
            f"<td class='hits'>{_html.escape(hits)}</td><td class='txt'>{txt}</td></tr>")
    return "\n".join(out) or "<tr><td colspan='6'>none</td></tr>"


def dominant_lang(sub):
    m = sub[["heb_ratio", "cyr_ratio", "lat_ratio"]].mean()
    return {"heb_ratio": "Hebrew", "cyr_ratio": "Russian",
            "lat_ratio": "English"}[m.idxmax()] if m.max() > 0 else "—"


scored_df = df[~df["category"].isin(EXCLUDED_CATEGORIES)]

# Per-channel summary
chan_rows = []
for ch, sub in scored_df.groupby("channel"):
    flags = (sub["flag_reason"] != "").sum()
    chan_rows.append(
        f"<tr><td>{_html.escape(str(ch))}</td><td>{sub['category'].iloc[0]}</td>"
        f"<td>{len(sub)}</td><td>{flags}</td>"
        f"<td>{100*flags/max(len(sub),1):.1f}%</td>"
        f"<td>{dominant_lang(sub)}</td>"
        f"<td>{sub['appearance_anomaly'].mean():.2f}</td></tr>")
chan_html = "\n".join(chan_rows)

kw_tab  = _rows(scored_df[scored_df["suspicion"] >= KEYWORD_MIN]
                .sort_values("suspicion", ascending=False), "suspicion")
app_tab = _rows(scored_df[scored_df["appearance_anomaly"] >= APPEARANCE_MIN]
                .sort_values("appearance_anomaly", ascending=False), "appearance_anomaly")

generated = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
n_kw  = int((scored_df["suspicion"] >= KEYWORD_MIN).sum())
n_app = int((scored_df["appearance_anomaly"] >= APPEARANCE_MIN).sum())

HTML = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<title>RecruitRadar-IL dashboard</title><style>
 body{{font-family:system-ui,Arial;margin:1.5rem;background:#0f1216;color:#e7eaee}}
 h1{{margin:0 0 .3rem}} .sub{{color:#9aa4b2;margin-bottom:1rem}}
 .cards{{display:flex;gap:1rem;margin-bottom:1rem;flex-wrap:wrap}}
 .card{{background:#1a1f27;border:1px solid #2a313c;border-radius:10px;padding:.8rem 1.1rem}}
 .card b{{font-size:1.6rem;display:block}}
 .tabs{{display:flex;gap:.4rem;margin:1rem 0 .5rem}}
 .tabs button{{background:#1a1f27;color:#e7eaee;border:1px solid #2a313c;border-radius:8px;
   padding:.45rem .9rem;cursor:pointer}} .tabs button.on{{background:#2d6cdf;border-color:#2d6cdf}}
 table{{border-collapse:collapse;width:100%;font-size:.86rem;background:#141821;border-radius:8px;overflow:hidden}}
 th,td{{border-bottom:1px solid #232a34;padding:.4rem .55rem;text-align:left;vertical-align:top}}
 th{{background:#1c222c;position:sticky;top:0;cursor:pointer}}
 td.txt{{max-width:620px;white-space:pre-wrap;direction:auto}} td.hits{{color:#f0b86e}}
 .panel{{display:none}} .panel.on{{display:block}}
</style></head><body>
<h1>RecruitRadar-IL — anomaly dashboard</h1>
<div class="sub">Generated {generated} · {len(scored_df)} scored messages ·
 {scored_df['channel'].nunique()} channels · news excluded. Leads to review by hand, not verdicts.</div>
<div class="cards">
 <div class="card"><b>{n_kw}</b>keyword anomalies</div>
 <div class="card"><b>{n_app}</b>appearance anomalies</div>
 <div class="card"><b>{int((scored_df['flag_reason']=='both').sum())}</b>flagged by both</div>
 <div class="card"><b>{scored_df['channel'].nunique()}</b>channels</div>
</div>
<div class="tabs">
 <button class="on" onclick="show('kw',this)">Keyword anomalies</button>
 <button onclick="show('app',this)">Appearance anomalies</button>
 <button onclick="show('ch',this)">Channels</button>
</div>
<div id="kw" class="panel on"><table><thead><tr><th>susp</th><th>channel</th><th>cat</th>
 <th>date</th><th>hits</th><th>text</th></tr></thead><tbody>{kw_tab}</tbody></table></div>
<div id="app" class="panel"><table><thead><tr><th>anomaly</th><th>channel</th><th>cat</th>
 <th>date</th><th>hits</th><th>text</th></tr></thead><tbody>{app_tab}</tbody></table></div>
<div id="ch" class="panel"><table><thead><tr><th>channel</th><th>cat</th><th>msgs</th>
 <th>flags</th><th>flag%</th><th>lang</th><th>avg anomaly</th></tr></thead>
 <tbody>{chan_html}</tbody></table></div>
<script>
 function show(id,btn){{for(const p of document.querySelectorAll('.panel'))p.classList.remove('on');
  document.getElementById(id).classList.add('on');
  for(const b of document.querySelectorAll('.tabs button'))b.classList.remove('on');btn.classList.add('on');}}
 document.querySelectorAll('th').forEach((th,i)=>th.onclick=()=>{{
  const tb=th.closest('table').querySelector('tbody');
  const rows=[...tb.rows].sort((a,b)=>{{const x=a.cells[i].innerText,y=b.cells[i].innerText;
   const nx=parseFloat(x),ny=parseFloat(y);return isNaN(nx)||isNaN(ny)?x.localeCompare(y):ny-nx;}});
  rows.forEach(r=>tb.appendChild(r));}});
</script></body></html>"""

dash_path = EXPORT_DIR / "dashboard.html"
dash_path.write_text(HTML, encoding="utf-8")
print(f"Dashboard → {dash_path}  ({n_kw} keyword + {n_app} appearance anomalies)")
print("Open it in a browser. Tabs: keyword / appearance / per-channel.")


## 13. Anonymized export & next steps

Export the combined flagged set (keyword **and** appearance anomalies)
**without** any raw identifier — sender is already pseudonymized. Safe to share
with collaborators / authorities. Raw data stays in `data/` (gitignored).


In [ ]:
export_cols = ["channel", "category", "date", "msg_id", "sender_hash",
               "flag_reason", "flag_score", "suspicion", "appearance_anomaly",
               "rule_score", "rule_hits", "text"]
out = EXPORT_DIR / f"review_queue_{datetime.now():%Y%m%d_%H%M}.csv"
flagged[export_cols].to_csv(out, index=False, encoding="utf-8-sig")
print("Wrote", out, f"({len(flagged)} rows: keyword + appearance anomalies)")
print("Reminder: this contains message TEXT. Strip it before any public release "
      "if the text could identify a victim/sender.")


### Where to go from here

1. **Labeling** — hand-verify the review queue → your first ground-truth set.
   Add real quotes from published news reports of recruitment cases as a
   positive seed set; sample random low-score posts as negatives.
2. **Weak supervision at scale** — port `RULES` into Snorkel labeling functions,
   train a label model, then a classifier on the de-noised labels.
3. **Text model** — fine-tune AlephBERT / HeBERT on the corpus.
4. **Behavioral model** — XGBoost on sender/channel features.
5. **Graph** — build the sender↔channel interaction graph; a GNN can catch
   coordinated accounts the text model misses.
6. **Validation** — hold-out set from media-reported cases never seen in
   training. Optimize for **precision**.

Close the Telegram client when done:


In [ ]:
if client is not None:
    await client.disconnect()
    print("Telegram client disconnected.")
else:
    print("No Telegram client was open.")
conn.close()
print("DB closed.")
